In [2]:
# notebook: 08_imf_fdi_2d_framework.ipynb
# ============================================================================
# CMVTS Extension — IMF FDI label, 2-D decision framework, sensitivity + figures
# ----------------------------------------------------------------------------
# Uses the IMF Financial Development Index CSV (Svirydzenka 2016, IMF WP/16/5)
# downloaded from data.imf.org, to build the OUTCOME LABEL for Section 5.3.
#
# Key finding (see notebook 05-07 for the predictor side):
#   FI/FID (depth/access) correlate strongly with macro-CMVTS (+0.85) -> circular.
#   FIE (efficiency) is INDEPENDENT of macro-CMVTS (+0.22, p=.58) -> valid label.
#   => We reframe the 1-D threshold into a 2-D decision grid:
#         axis 1 = transferability (macro-CMVTS)
#         axis 2 = local absorptive capacity (FIE)
#
# Figure spec (house style):
#   seaborn greyscale, NO caption text in the image, dpi=600, save BOTH png+pdf,
#   legend (if any) placed at the BOTTOM.
# ============================================================================

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------------------------------------------------------
# 0. CONFIG
# ----------------------------------------------------------------------------
IMF_CSV = "dataset_2026-09-17T04_43_36.338606496Z_DEFAULT_INTEGRATION_IMF.MCM_FDI_1.0.0.csv"
FIG_DIR = "."          # where png/pdf are written
FIG_DPI = 600

# macro-CMVTS (equal-weight) from notebook 07, and realized divergence Y from NB03.
MACRO_CMVTS = {
    "Indonesia":0.6734,"Thailand":0.7762,"Viet Nam":0.6842,"Philippines":0.5650,
    "Bangladesh":0.5561,"Cambodia":0.6034,"Nepal":0.5874,"Pakistan":0.4412,"Lao PDR":0.5434,
}
Y_DIVERGENCE = {
    "Indonesia":0.0708,"Thailand":0.0092,"Viet Nam":0.0000,"Philippines":0.1809,
    "Bangladesh":0.2110,"Cambodia":0.0810,"Nepal":0.0332,"Pakistan":0.2192,"Lao PDR":0.0753,
}
TARGETS = list(MACRO_CMVTS.keys())

# IMF country-name -> our label
CMAP = {
    "Korea, Republic of":"Korea, Rep.","Indonesia":"Indonesia","Thailand":"Thailand",
    "Vietnam":"Viet Nam","Philippines":"Philippines","Bangladesh":"Bangladesh",
    "Cambodia":"Cambodia","Nepal":"Nepal","Pakistan":"Pakistan",
    "Lao People's Democratic Republic":"Lao PDR",
}
# FI-family sub-indices (credit-relevant). FM (markets) is irrelevant to credit transfer.
WANT = {
    "Financial Institutions Efficiency Index":"FIE",   # label core (independent)
    "Financial Institutions Depth Index":"FID",        # overlaps predictor D4
    "Financial Institutions Access Index":"FIA",       # overlaps predictor B7/B8
    "Financial Institutions Index":"FI",               # aggregate (contains FIA)
    "Financial Development Index":"FD",                 # broadest
}

# ----------------------------------------------------------------------------
# 1. Load IMF FDI, extract FI-family sub-indices at latest year (2020 -> 2019 -> 2018)
# ----------------------------------------------------------------------------
raw = pd.read_csv(IMF_CSV, low_memory=False)
sub = raw[raw["COUNTRY"].isin(CMAP) & raw["INDICATOR"].isin(WANT)].copy()
sub["economy"] = sub["COUNTRY"].map(CMAP)
sub["idx"]     = sub["INDICATOR"].map(WANT)

def latest_value(row):
    for y in ["2020","2019","2018"]:
        if y in row and pd.notna(row[y]):
            return row[y], y
    return np.nan, None

vals = sub.apply(lambda r: pd.Series(latest_value(r), index=["value","year"]), axis=1)
sub = pd.concat([sub[["economy","idx"]], vals], axis=1)

ORDER = ["Korea, Rep."] + TARGETS
label = sub.pivot_table(index="economy", columns="idx", values="value").reindex(ORDER)
label = label[["FIE","FID","FIA","FI","FD"]]
print("=== IMF FDI FI-family sub-indices (latest<=2020) ===")
print(label.round(4).to_string())
print("\nCoverage / 10:"); print(label.notna().sum().to_string())
print("Years used:", sub["year"].value_counts().to_dict())

# ----------------------------------------------------------------------------
# 2. Which sub-index is a VALID (independent) label? correlate with macro-CMVTS
# ----------------------------------------------------------------------------
mc = pd.Series(MACRO_CMVTS); yv = pd.Series(Y_DIVERGENCE)
tab = pd.DataFrame({"macroCMVTS":mc,"Y_divergence":yv}).join(label.loc[TARGETS])
print("\n=== Label independence check (vs macro-CMVTS) ===")
for col in ["FIE","FI","FID"]:
    r,p   = stats.pearsonr(tab["macroCMVTS"], tab[col])
    rs,ps = stats.spearmanr(tab["macroCMVTS"], tab[col])
    tag = "INDEPENDENT (valid label)" if abs(rs)<0.4 else "correlated (circular risk)"
    print(f"  {col}: Pearson {r:+.3f}(p={p:.3f}) Spearman {rs:+.3f}(p={ps:.3f})  -> {tag}")
print("Decision: FIE is the label (independent of the transferability axis).")

# ----------------------------------------------------------------------------
# 3. 2-D decision grid: transferability (macro-CMVTS) x absorptive capacity (FIE)
# ----------------------------------------------------------------------------
grid = pd.DataFrame({"T":mc, "A":tab["FIE"]}).loc[TARGETS]
t_med, a_med = grid["T"].median(), grid["A"].median()

def quadrant(t, a, tc=t_med, ac=a_med):
    hi_t, hi_a = t>=tc, a>=ac
    if hi_t and hi_a:      return "Q1 direct-transfer"
    if hi_t and not hi_a:  return "Q3 transfer+monitor"
    if (not hi_t) and hi_a:return "Q2 redevelop/good-soil"
    return "Q4 local-redevelop"

grid["quadrant"] = [quadrant(grid.loc[c,"T"], grid.loc[c,"A"]) for c in TARGETS]
print(f"\n=== 2-D grid (median split: T={t_med:.3f}, A={a_med:.3f}) ===")
print(grid.sort_values(["quadrant","T"],ascending=[True,False]).round(4).to_string())

# ----------------------------------------------------------------------------
# 4. Sensitivity — split-rule variation + jitter flip-rate
# ----------------------------------------------------------------------------
splits = {
    "median":      (grid["T"].median(),      grid["A"].median()),
    "mean":        (grid["T"].mean(),        grid["A"].mean()),
    "tercile-hi":  (grid["T"].quantile(2/3), grid["A"].quantile(2/3)),
    "tercile-lo":  (grid["T"].quantile(1/3), grid["A"].quantile(1/3)),
}
srule = pd.DataFrame({name:[quadrant(grid.loc[c,"T"],grid.loc[c,"A"],tc,ac)[:2]
                            for c in TARGETS]
                      for name,(tc,ac) in splits.items()}, index=TARGETS)

rng = np.random.default_rng(0)
sT, sA = grid["T"].std()*0.15, grid["A"].std()*0.15
base_q = {c:quadrant(grid.loc[c,"T"],grid.loc[c,"A"])[:2] for c in TARGETS}
N=2000; flips={c:0 for c in TARGETS}
for _ in range(N):
    for c in TARGETS:
        if quadrant(grid.loc[c,"T"]+rng.normal(0,sT),
                    grid.loc[c,"A"]+rng.normal(0,sA))[:2] != base_q[c]:
            flips[c]+=1
grid["flip_rate"] = [flips[c]/N for c in TARGETS]
srule["flip_rate"] = grid["flip_rate"].round(3)
print("\n=== Sensitivity: quadrant under split rules + jitter flip-rate ===")
print(srule.sort_values("flip_rate",ascending=False).to_string())

grid.to_csv("cmvts_2d_grid.csv")
label.to_csv("imf_fdi_label.csv")

# ============================================================================
# FIGURES — seaborn greyscale, no caption in image, dpi 600, png+pdf, legend bottom
# ============================================================================
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.size":11, "axes.edgecolor":"0.2", "axes.linewidth":0.8,
    "grid.color":"0.85", "figure.dpi":120,
})
GREY = {"Q1":"0.15","Q2":"0.45","Q3":"0.65","Q4":"0.85"}   # greyscale by quadrant

def save(fig, name):
    for ext in ("png","pdf"):
        fig.savefig(f"{FIG_DIR}/{name}.{ext}", dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {name}.png / {name}.pdf")

# ---- FIGURE 1: 2-D decision grid scatter -----------------------------------
fig, ax = plt.subplots(figsize=(6.4,5.4))
markers = {"Q1":"o","Q2":"s","Q3":"^","Q4":"D"}
for q in ["Q1","Q2","Q3","Q4"]:
    m = grid["quadrant"].str.startswith(q)
    ax.scatter(grid.loc[m,"T"], grid.loc[m,"A"],
               s=90, c=GREY[q], marker=markers[q],
               edgecolors="black", linewidths=0.8, label=q, zorder=3)
# median split lines
ax.axvline(t_med, color="0.3", ls="--", lw=0.9, zorder=1)
ax.axhline(a_med, color="0.3", ls="--", lw=0.9, zorder=1)
# country labels
for c in TARGETS:
    ax.annotate(c, (grid.loc[c,"T"], grid.loc[c,"A"]),
                xytext=(4,4), textcoords="offset points", fontsize=8.5, color="0.1")
ax.set_xlabel("Transferability  (macro-CMVTS)")
ax.set_ylabel("Local absorptive capacity  (FIE)")
# legend at bottom
ax.legend(title=None, loc="upper center", bbox_to_anchor=(0.5,-0.12),
          ncol=4, frameon=False, handletextpad=0.4, columnspacing=1.2)
fig.tight_layout()
save(fig, "fig_2d_decision_grid")

# ---- FIGURE 2: label independence (macro-CMVTS vs FIE vs FI) ----------------
fig, axes = plt.subplots(1,2, figsize=(9.2,4.4), sharey=False)
for ax, col, title in zip(axes, ["FIE","FI"],
                          ["FIE (independent label)","FI (correlated → circular)"]):
    ax.scatter(tab["macroCMVTS"], tab[col], s=70, c="0.35",
               edgecolors="black", linewidths=0.7, zorder=3)
    # regression line (greyscale)
    b,a = np.polyfit(tab["macroCMVTS"], tab[col], 1)
    xs = np.linspace(tab["macroCMVTS"].min(), tab["macroCMVTS"].max(), 50)
    ax.plot(xs, a+b*xs, color="0.1", lw=1.2, ls="-", zorder=2)
    rs,ps = stats.spearmanr(tab["macroCMVTS"], tab[col])
    ax.set_title(f"{title}\nSpearman = {rs:+.2f} (p={ps:.2f})", fontsize=10)
    ax.set_xlabel("Transferability (macro-CMVTS)")
    ax.set_ylabel(col)
fig.tight_layout()
save(fig, "fig_label_independence")

# ---- FIGURE 3: sensitivity flip-rate bar ------------------------------------
fig, ax = plt.subplots(figsize=(6.4,4.2))
fr = grid["flip_rate"].sort_values()
bars = ax.barh(fr.index, fr.values, color="0.55", edgecolor="black", linewidth=0.7)
# shade robust (<0.1) darker
for b, v in zip(bars, fr.values):
    b.set_color("0.25" if v < 0.10 else "0.7")
    b.set_edgecolor("black")
ax.axvline(0.10, color="0.2", ls="--", lw=0.9)
ax.set_xlabel("Quadrant flip rate under 15%-SD jitter")
ax.set_ylabel("")
fig.tight_layout()
save(fig, "fig_sensitivity_fliprate")

print("\nAll figures written at dpi=600 (png+pdf), greyscale, legend at bottom, no captions.")
print("Data: cmvts_2d_grid.csv, imf_fdi_label.csv")

=== IMF FDI FI-family sub-indices (latest<=2020) ===
idx            FIE    FID    FIA     FI     FD
economy                                       
Korea, Rep.  0.716  0.889  0.662  0.855  0.832
Indonesia    0.628  0.157  0.401  0.402  0.363
Thailand     0.681  0.639  0.616  0.717  0.737
Viet Nam     0.735  0.333  0.162  0.403  0.389
Philippines  0.666  0.195  0.235  0.357  0.371
Bangladesh   0.672  0.107  0.148  0.283  0.244
Cambodia     0.601  0.252  0.248  0.370  0.190
Nepal        0.611  0.278  0.330  0.420  0.214
Pakistan     0.700  0.077  0.166  0.285  0.229
Lao PDR      0.577  0.054  0.157  0.239  0.168

Coverage / 10:
idx
FIE    10
FID    10
FIA    10
FI     10
FD     10
Years used: {'2020': 50}

=== Label independence check (vs macro-CMVTS) ===
  FIE: Pearson +0.149(p=0.703) Spearman +0.217(p=0.576)  -> INDEPENDENT (valid label)
  FI: Pearson +0.838(p=0.005) Spearman +0.850(p=0.004)  -> correlated (circular risk)
  FID: Pearson +0.833(p=0.005) Spearman +0.867(p=0.002)  -> corre